In [22]:
import torch
from torch import nn
import matplotlib.pyplot as plt

## **Layer Norm** (post vs pre and RMSNorm)

In [13]:
from src.model.ffn import FeedForwardBlock
from src.model.ln import LayerNormalization
ffn_1, ffn_2 = FeedForwardBlock(20), FeedForwardBlock(20)

In [83]:
ln = LayerNormalization(64, epsilon=1e-10)
for scale in [0.01, 1.0, 100.0]:
    v = torch.randn(1, 64) * scale
    print(f"in: {v.norm():8.3f}   out: {ln(v).norm():8.3f}")
    # std = 1, and mean = 0. length of vector = sqrt(dims)

in:    0.066   out:    8.000
in:    7.791   out:    8.000
in:  808.945   out:    8.000


In [93]:
d = 64
N = 4096

In [97]:
class LN(nn.Module):
    def __init__(self, pre = False, alpha=1):
        super().__init__()
        self.ffn = FeedForwardBlock(d)
        self.ln = LayerNormalization(d, epsilon=1e-10)
        self.pre = pre
        self.alpha = alpha
    def forward(self, h): # we aren't really training so alpha is what controls the weight of the ffn. if the model wants more of ffn, it will increase the magnitude of ffn in post-LN.
        if self.pre:
            return h + self.alpha*self.ffn(self.ln(h))
        else:
            return self.ln(h + self.alpha*self.ffn(h))
pre_LN = LN(pre=True)
post_LN = LN(pre=False)

In [103]:
@torch.no_grad()
def run(pre, alpha=1.0, L=1000, seed=0):
    torch.manual_seed(seed)
    nn = [LN(pre, alpha) for _ in range(L)]
    X = torch.randn(N, d)
    hs, h = [X], X
    for l in nn:
        h = l(h)
        hs.append(h)
    return hs

pre  = run(True)
post = run(False)

In [104]:
def mean_norm(h):
    return h.norm(dim=1).mean().item()

for l in [0, 1, 10, 25, 50, 250, 1000]:
    print(f"l={l:3d}   pre: {mean_norm(pre[l]):8.2f}   post: {mean_norm(post[l]):8.2f}")

l=  0   pre:     7.96   post:     7.96
l=  1   pre:     7.96   post:     8.00
l= 10   pre:     8.02   post:     8.00
l= 25   pre:     8.08   post:     8.00
l= 50   pre:     8.21   post:     8.00
l=250   pre:     9.24   post:     8.00
l=1000   pre:    12.37   post:     8.00


In [106]:
for l in [0, 1, 10, 25, 50, 250, 1000]:
    print(f"l={l:3d}   pre: {torch.norm(pre[l]):8.2f}   post: {torch.norm(post[l]):8.2f}")

l=  0   pre:   511.75   post:   511.75
l=  1   pre:   511.77   post:   512.00
l= 10   pre:   515.18   post:   512.00
l= 25   pre:   519.25   post:   512.00
l= 50   pre:   527.38   post:   512.00
l=250   pre:   593.96   post:   512.00
l=1000   pre:   794.96   post:   512.00


In [105]:
import torch.nn.functional as F
def cos_consec(a, b):
    return F.cosine_similarity(a, b, dim=1).mean().item()

for l in [0, 1, 10, 25, 50, 250, 1000]:
    print(f"l={l:3d}   1-cos pre: {1-cos_consec(pre[l-1], pre[l]):.5f}"
          f"   post: {1-cos_consec(post[l-1], post[l]):.5f}")

l=  0   1-cos pre: 0.36722   post: 0.50743
l=  1   1-cos pre: 0.00066   post: 0.00866
l= 10   1-cos pre: 0.00073   post: 0.00071
l= 25   1-cos pre: 0.00069   post: 0.00068
l= 50   1-cos pre: 0.00068   post: 0.00069
l=250   1-cos pre: 0.00053   post: 0.00067
l=1000   1-cos pre: 0.00033   post: 0.00080


In [110]:
@torch.no_grad()
def perturb_test(pre, alpha=1.0, L=1000, seed=0):
    torch.manual_seed(seed)
    x  = torch.randn(512, d)
    x2 = x + 0.01 * torch.randn(512, d)      # tiny nudge to the input
    blocks = [LN(pre, alpha) for _ in range(L)]
    h, h2, out = x, x2, []
    for l, b in enumerate(blocks):
        h, h2 = b(h), b(h2)
        out.append((l+1, (h - h2).norm(dim=1).mean().item()))
    return out

for l, gap in perturb_test(False):
    if l in [1, 5, 10, 20, 1000]: print(f"post  l={l:3d}  gap: {gap:.3e}")
print("\n")
for l, gap in perturb_test(True):
    if l in [1, 5, 10, 20, 1000]: print(f"pre   l={l:3d}  gap: {gap:.3e}")

post  l=  1  gap: 7.954e-02
post  l=  5  gap: 7.963e-02
post  l= 10  gap: 7.970e-02
post  l= 20  gap: 7.980e-02
post  l=1000  gap: 7.605e-02


pre   l=  1  gap: 7.946e-02
pre   l=  5  gap: 7.965e-02
pre   l= 10  gap: 7.991e-02
pre   l= 20  gap: 8.038e-02
pre   l=1000  gap: 1.197e-01
